Exploratory notebook: final presentation figures are generated by `python3 run.py` and recorded in `analysis/current/results.json`. Some narrative counts below predate the final date-filter, pricing and cost changes. Use the current report for claims; run the raw workflow and download with `--all` before using reviews/calendars.

# Data overview & plan

All data: Inside Airbnb, Vancouver, snapshot **15 June 2026** (CC BY 4.0). City of Vancouver only, 23 neighbourhoods.

---

## 1. `listings.csv.gz`: one row per listing (main file)
**6,509 listings × 90 columns**, one-day snapshot.

| What's in it | Key columns |
|---|---|
| Location | `neighbourhood_cleansed`, lat/long |
| What it is | `property_type`, `room_type`, `bedrooms`, `accommodates` |
| Price | `price` (median $318/night) |
| Rules | `minimum_nights` |
| Demand | `number_of_reviews_ltm` (last 12 months), `reviews_per_month`, `review_scores_rating` |
| Inside Airbnb estimates | `estimated_occupancy_l365d`, `estimated_revenue_l365d` |
| Host | `calculated_host_listings_count` |
| Regulation | `license` |

- **Mix:** Downtown has 1,534 listings, nearly 4× the next neighbourhood (Kitsilano, 440). 82% entire homes, 18% private rooms.
- **Answers:** where (neighbourhood), what type (condo / house / suite, bedrooms), how much revenue each type earns.
- **Problems:**
  - 10% have no price.
  - 30% require 90+ night stays (not short-term under BC/Vancouver rules, which we drop).
  - 13% are missing bedrooms; `host_since` is empty everywhere.
  - `price` is a quote for the World Cup week, so it may be inflated.
  - Our revenue model gives the same numbers as Inside Airbnb's `estimated_revenue_l365d` (Spearman ρ ≈ 1.0), so it is **not** an independent check.
- **Regulation angle:** 17.5% have no licence and 8.6% claim exempt. 29% of listings belong to hosts with 3+ listings (commercial operators).

## 2. `reviews.csv.gz`: one row per guest review
**323,903 reviews of 5,376 listings, 2010 to June 2026.** Columns: `listing_id`, `date`, reviewer name, comment text.

- **Answers:** demand trend and what to expect in 2027.
  - Growth: 47k reviews (2023), 60k (2024), 73k (2025); Jan–Apr 2026 is up ~48% on Jan–Apr 2025.
  - Seasonality: summer ≈ 3× winter.
  - Neighbourhood-level growth or decline.
- **Problems:**
  - A review is a proxy for a stay; only ~half of guests leave one.
  - The May 2026 spike (12.1k vs 7.1k in May 2025) is probably the World Cup and won't repeat in 2027.
  - 41 reviews are dated after the snapshot.
  - Never show reviewer names.
- **Bonus:** comment text could support a guest-satisfaction angle (optional).

## 3. `calendar.csv.gz`: one row per listing per night
**2.38M rows: 6,525 listings × 365 nights, 15 June 2026 to July 2027.** Columns: `listing_id`, `date`, `available`, min/max nights.

- **Answers:** forward-looking view, i.e. how booked-up 2026–27 already looks and which months are tight.
- **Problems:**
  - **No prices** in this snapshot.
  - "Not available" could mean booked or host-blocked.
  - 6.4% of listings are blocked for the whole year.
- **Verdict:** supporting evidence only, not a revenue source.

## 4. `neighbourhoods.geojson`: map boundaries
24 neighbourhood shapes (names only). Used for the "where we'd buy" map.

## 5. Outside data (brief requires at least one)
- **Done: purchase prices.** City of Vancouver property tax report 2026 (BC Assessment values): median value per neighbourhood × condo/house → `data/external/property_values_2026.csv` (built by `scripts/build_property_values.py`). Condos scaled by bedrooms (1-bed 0.85×, 2-bed 1.15×, 3+ 1.5×). Suites excluded (can't buy without the house). Strathcona has no homes in the tax data, so it drops out.
- **Optional:** CMHC rental market data (vacancy → opt-out risk; long-term-rent fallback).

---

## How this maps to the presentation

| Question from the brief | Main file | Support |
|---|---|---|
| **Where** to buy | listings (revenue by neighbourhood) | geojson map |
| **What type / mix** | listings (property type × bedrooms) | |
| **How many** units | listings + purchase prices → `few_vs_many` | |
| **What to expect in 2027** | listings (base revenue) | reviews (trend, seasonality), calendar (forward bookings) |
| **Biggest risks** | listings (licences, commercial hosts) | reviews (World Cup spike), outside data (rent fallback) |

## Team split
1. **Portfolio**: check the recommended portfolio, few vs many, sensitivity; stress-test assumptions.
2. **Review trend + seasonality**: for the 2027 outlook slide.
3. **Regulation & risk**: licences, commercial hosts, World Cup pricing.
4. **Charts & map**.

In [1]:
%load_ext autoreload
%autoreload 2
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# make the project's src/ importable from inside notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config, prep, portfolio

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)
%matplotlib inline

In [ ]:
# Run `python run.py` from the repo root first if outputs/ is empty or you changed config.py
T = config.TABLES
df = pd.read_csv(config.PROCESSED / "listings_clean.csv", low_memory=False)   # cleaned listings + our revenue columns
df.head()

## Ready-to-use tables
| Variable | What it is |
|---|---|
| `df` | cleaned listings (4,447), one row per listing |
| `seg` | neighbourhood × type × bedrooms, active entire homes: count, nightly price, occupancy, revenue p25/median/p75 |
| `uw` | `seg` + purchase price, costs, NOI and **net yield** (buyable types only) |
| `port` | recommended $10M portfolio (one row per unit) |
| `fvm` | few high-value vs many modest |
| `sens` | low / base / high / 2-months-offline range |
| `key` | headline numbers |
| `rev_monthly` | reviews per month (demand trend + seasonality) |
| `cal` | per listing: share of next 365 nights unavailable |

In [ ]:
seg  = pd.read_csv(T / "segments.csv", dtype={"bedrooms_grp": str})
uw   = pd.read_csv(T / "segments_underwritten.csv", dtype={"bedrooms_grp": str})
port = pd.read_csv(T / "recommended_portfolio.csv", dtype={"bedrooms_grp": str})
fvm  = pd.read_csv(T / "few_vs_many.csv")
sens = pd.read_csv(T / "sensitivity.csv")
key  = json.loads((config.OUTPUTS / "key_numbers.json").read_text())
key

### Recommended portfolio (grouped)

In [ ]:
(port.groupby(["neighbourhood_cleansed", "bucket", "bedrooms_grp"])
     .agg(units=("n", "size"), price=("price_cad", "first"), revenue=("rev_median", "first"),
          noi_each=("noi", "first"), net_yield=("net_yield", "first"), comparables=("n", "first"))
     .sort_values("units", ascending=False))

In [ ]:
fvm

In [ ]:
sens

### Top segments by yield (what the portfolio picks from)

In [ ]:
cols = ["neighbourhood_cleansed", "bucket", "bedrooms_grp", "n", "adr_median", "occ_median",
        "rev_median", "price_cad", "noi", "net_yield", "share_licensed_est"]
uw[uw["n"] >= 5].sort_values("net_yield", ascending=False)[cols].head(20)

### Reviews over time (demand trend + seasonality)

In [ ]:
reviews = pd.read_csv(config.RAW / "reviews.csv.gz", usecols=["listing_id", "date"])   # no reviewer names
reviews["date"] = pd.to_datetime(reviews["date"], format="%Y-%m-%d")
rev_monthly = reviews.set_index("date").resample("MS").size().rename("reviews")
rev_monthly["2022":].plot(figsize=(10, 3), title="Reviews per month (proxy for stays)"); plt.show()

### Calendar (forward availability, no prices)

In [ ]:
c = pd.read_csv(config.RAW / "calendar.csv.gz", usecols=["listing_id", "date", "available"])
c["date"] = pd.to_datetime(c["date"], format="%Y-%m-%d")
c["unavail"] = c["available"].eq("f")
cal = c.groupby("listing_id")["unavail"].mean().rename("share_unavailable").reset_index()
print(f"fully blocked all year: {(cal.share_unavailable >= 0.999).mean():.1%}")
c.groupby(c["date"].dt.to_period("M"))["unavail"].mean().plot(kind="bar", figsize=(10, 3),
    title="Share of nights unavailable, by month ahead"); plt.show()

## Analysis
_Start here._